# Maryland Pipeline — OpenAI Version


 ## Imports & Configuration


In [ ]:
import openai
import requests          
from bs4 import BeautifulSoup  
import pandas as pd      
import json              
import re                
import time              
import os
import logging
import sys
from datetime import datetime
from functools import wraps
from dotenv import load_dotenv  

load_dotenv()

# OpenAI credentials from .env
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# OpenAI model to use
OPENAI_MODEL = "gpt-4o-mini"

# Knack API — Maryland's backend database that powers businessexpress.maryland.gov
KNACK_URL = 'https://us-east-1-renderer-read.knack.com/v1/scenes/scene_1/views/view_17/records'
KNACK_HEADERS = {
    'X-Knack-Application-Id': '594ac6010f1b2d4e1e14a3b8',
    'X-Knack-REST-API-Key':    'renderer'                   
}

# Retry settings for HTTP calls
MAX_RETRIES        = 3
RETRY_BACKOFF_BASE = 2  # wait = base ** attempt → 1s, 2s, 4s

# OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Shared session — reuses TCP connections across all scrape calls (faster, less overhead)
session = requests.Session()

# Fresh timestamped log file every run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_filename  = f"maryland_pipeline_openai_{run_timestamp}.log"

# force=True ensures a clean reconfiguration every run even without restarting the kernel
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(log_filename, mode='w', encoding='utf-8')
    ],
    force=True  # override any existing handlers from a previous run in the same kernel session
)
logger = logging.getLogger(__name__)

# Suppress noisy HTTP request lines from openai/httpx
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Retry decorator — wraps any function with exponential backoff on request failures
# 4xx errors (except 429 Too Many Requests) are not retried — they won't recover
def with_retry(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        last_exc = None
        for attempt in range(MAX_RETRIES):
            try:
                return func(*args, **kwargs)
            except requests.HTTPError as exc:
                status = exc.response.status_code if exc.response is not None else None
                if status is not None and 400 <= status < 500 and status != 429:
                    raise
                last_exc = exc
            except requests.RequestException as exc:
                last_exc = exc
            wait = RETRY_BACKOFF_BASE ** attempt
            logger.warning("Attempt %d/%d failed: %s — retrying in %ds", attempt + 1, MAX_RETRIES, last_exc, wait)
            time.sleep(wait)
        raise last_exc
    return wrapper

logger.info("=" * 60)
logger.info("Maryland Pipeline Run (OpenAI) — %s", run_timestamp)
logger.info("=" * 60)
logger.info("Config loaded | Model: %s | Session ready | Retry: %dx", OPENAI_MODEL, MAX_RETRIES)


## Fetch All Programs from Knack API


In [ ]:
programs_raw = []

params = {
    'format':        'both',
    'rows_per_page':  50,       
    'sort_field':    'field_21',
    'sort_order':    'asc'       
}

page        = 1
total_pages = 1

logger.info("Fetching programs from Knack API...")

while page <= total_pages:
    params['page'] = page  

    response = session.get(KNACK_URL, headers=KNACK_HEADERS, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()
    programs_raw.extend(data['records'])  
    total_pages = data['total_pages']

    logger.info("  Page %d/%d — %d records", page, total_pages, len(data['records']))
    page += 1         
    time.sleep(0.5)

logger.info("Total programs fetched: %d", len(programs_raw))


## Extract and Clean URLs from Raw Records

In [ ]:
programs = []
skipped  = 0

for record in programs_raw:
    all_found_urls = re.findall(r'href="([^"]+)"', record.get('field_23', ''))
    detail_url     = all_found_urls[0].strip() if all_found_urls else None

    if not detail_url:
        skipped += 1
        continue

    programs.append({
        'id':                 record.get('id', ''),
        'name':               record.get('field_21', '').strip(),
        'type':               record.get('field_22', '').strip(),
        'category':           record.get('field_27', '').strip(),
        'url':                detail_url,
        'all_extracted_urls': all_found_urls
    })

multi_url_count = sum(1 for p in programs if len(p['all_extracted_urls']) > 1)
logger.info("Programs with valid URLs: %d (skipped: %d) | multi-URL: %d", len(programs), skipped, multi_url_count)


## Prompts for Claude
Prompts given to Claude before sending any program data.

In [ ]:
# Main extraction prompt — used for every program's detail page
system_prompt = """You are extracting structured data from a government funding opportunity page. Extract every field below.
If a field cannot be found on the page, use 'Not specified' never invent data.

Fields to extract:

1.  title: Full program name.
2.  opportunity_type: Best match from: Grant, Loan, Tax Credit, Forgivable Loan, Accelerator,
    Incubator, Fellowship, Competition, Pro Bono, Internship, SaaS Credit, Stipend,
    Mentorship, Workshop, Legislative Initiative.
3.  summary: 2 sentences max. Key value and purpose of the opportunity.
4.  description: 5-10 sentences. Objectives, benefits, and eligibility overview.
5.  sponsor: The government agency or entity offering this opportunity.
6.  sponsor_website: The sponsor primary website URL. 'Not specified' if not found.
7.  logo_url: Logo image URL from the page. 'No logo URL found' if not present.
8.  direct_application_url: Direct application or submission link. 'Not specified' if not found.
9.  award_value: The single best dollar amount an individual applicant can receive.
    Use ONLY amounts from the verified dollar lines provided.
    IMPORTANT — only include dollar amounts that represent what an applicant receives.
    EXCLUDE amounts that are administrative thresholds, compliance requirements,
    contract registration limits, or EFT/payment processing minimums.
    If page shows total fund AND individual award amount, use the individual amount.
    If only a total fund size or program cap exists, write 'Varies'.
    If no award amount exists at all, write 'Not specified'. Never invent amounts.
10. cash_award: Cash portion if separate from total award. 'Not specified' if not found.
11. award_amounts: List of dollar amounts that represent actual award values with context:
    [{"amount": "$50,000", "context": "maximum individual grant award"}]
    DO NOT include compliance thresholds or EFT registration limits.
    Empty list if no actual award amounts found.
12. date_posted: Posting date MM-DD-YYYY. 'Not specified' if not found.
13. deadline: Application deadline MM-DD-YYYY. If rolling write 'Rolling'. Never empty.
14. rolling: 'Yes' if rolling basis. Otherwise 'No'.
15. global_opportunity: 'Yes' if globally available. Otherwise 'No'.
16. location: Geographic focus or eligible regions. 'Not specified' if not found.
17. fee_required: 'No' if no fee. If yes: 'Yes - $X'.
18. cost_to_participate: 'No' if no cost. If yes: 'Yes - $X'.
19. equity_percentage: 'No' if no equity. If yes: 'Yes - X%'.
20. safe_note: 'No' if no SAFE note. If yes: 'Yes - details'.
21. contact_names: Contact names if listed. 'Not specified' if none.
22. contact_email: Contact email if available. 'Not specified' if none.
23. contact_phone: Contact phone numbers if listed. 'Not specified' if none.
24. industry: Industries relevant to this opportunity. 'Not specified' if not found.
25. tags: List of keyword tags.
26. areas_of_focus: List from Capital, Networks, Capacity Building. Empty list if none apply.
27. eligibility: List of eligibility requirements as strings.
28. sdg_alignment: Applicable UN SDGs (e.g. 'SDG 8 Decent Work and Economic Growth').

Return valid JSON only."""

# Secondary URL prompt — used when a program page has additional links to analyze
secondary_system_prompt = """You are an AI assistant tasked with analyzing a secondary URL related to a primary funding program.
Your goal is to categorize the content of this secondary page and extract key information.
Focus on understanding its relationship to the primary program.

Fields to extract:
1.  type: Categorize the secondary link. Choose from: 'Related Program', 'Application Portal',
    'Resource Guide', 'Eligibility Details', 'Partner Website', 'General Information',
    'Other Funding Opportunity', 'Contact Page', 'Login/Account Page', 'Other'.
2.  summary: A brief 1-2 sentence description of what this page offers.
3.  details: A dictionary containing any specific relevant details found on this page.

Return valid JSON only."""

logger.info("OpenAI credentials loaded | Model: %s", OPENAI_MODEL)

In [ ]:
def parse_openai_json(raw: str) -> dict:
    """Extract JSON from OpenAI's response — handles raw JSON or ```json``` wrapped blocks."""
    raw = raw.strip()
    if not raw:
        raise json.JSONDecodeError("Empty response from OpenAI", "", 0)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL) or \
            re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        return json.loads(match.group(1) if match.lastindex else match.group(0))
    raise json.JSONDecodeError("No JSON found in OpenAI response", raw, 0)


@with_retry
def scrape_page(url: str) -> tuple:
    """Fetch a page and return (full_text, dollar_lines). Retries on network failures."""
    response  = session.get(url.strip(), timeout=10)
    response.raise_for_status()
    soup      = BeautifulSoup(response.text, 'html.parser')
    full_text = soup.get_text(separator='\n')
    full_text = full_text.encode('ascii', 'ignore').decode('ascii')
    dollar_lines = [
        line.strip() for line in full_text.split('\n')
        if '$' in line and line.strip()
    ]
    return full_text, dollar_lines


def extract_with_openai(program_name: str, page_text: str, dollar_lines: list) -> tuple:
    """Send page content to OpenAI and return (extracted_dict, tokens_used)."""
    dollar_context = '\n'.join(
        [f'[{i+1}] {l}' for i, l in enumerate(dollar_lines)]
    ) if dollar_lines else 'No dollar amounts found.'

    user_prompt = f"""Program Name: {program_name}

Verified dollar lines from page:
{dollar_context}

Page Text Content:
{page_text[:15000]}

Extract as JSON. For award_amounts, only include dollar amounts that represent what an applicant receives — exclude fund totals, compliance thresholds, and EFT registration limits.
For award_value, pick the individual award amount not the total fund size."""

    response  = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.1,
        max_tokens=2500,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt}
        ]
    )
    raw_content = response.choices[0].message.content or ""
    extracted   = parse_openai_json(raw_content)
    tokens      = response.usage.prompt_tokens + response.usage.completion_tokens
    return extracted, tokens


def analyze_secondary_url(program_name: str, primary_url: str, sec_url: str) -> dict:
    """Fetch and analyze a secondary URL. Always returns a result dict, never raises."""
    result = {
        'url': sec_url, 'status': 'failed', 'reason': 'unknown',
        'type': 'Not specified', 'summary': 'Not specified',
        'details': {}, 'tokens_used': 0
    }
    try:
        sec_text, sec_dollar_lines = scrape_page(sec_url)
        sec_user_prompt = f"""Analyze the content of this secondary page for the program '{program_name}'.
Primary URL: {primary_url}
Secondary URL to analyze: {sec_url}

Page Text Content:
{sec_text[:15000]}

Extracted dollar lines (if any):
{json.dumps(sec_dollar_lines, indent=2)}

Extract as JSON."""
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.1,
            max_tokens=1000,
            messages=[
                {'role': 'system', 'content': secondary_system_prompt},
                {'role': 'user',   'content': sec_user_prompt}
            ]
        )
        raw_content   = response.choices[0].message.content or ""
        sec_extracted = parse_openai_json(raw_content)
        sec_tokens    = response.usage.prompt_tokens + response.usage.completion_tokens
        result.update({
            'status':      'success',
            'reason':      'page fetched and analyzed',
            'type':        sec_extracted.get('type',    'Other'),
            'summary':     sec_extracted.get('summary', 'Not specified'),
            'details':     sec_extracted.get('details', {}),
            'tokens_used': sec_tokens
        })
    except Exception as e:
        result['reason'] = str(e)
        logger.warning("    secondary scrape/analyze error: %s", e)
    return result


logger.info("Helper functions defined | @with_retry active on scrape_page")

## Main Pipeline Loop

For each program, we visit the first URL and extract based on the prompt.
BeautifulSoup helps scan the text for every line containing a `$`.
During testing we noticed the agent hallucinating numbers so we included this as a safety net.

In [ ]:
clean_records   = []  # records ready for the database
flagged_records = []  # records that need human review 
errors          = []  # programs that failed entirely

# For sample run use programs[:5]
run_list = programs

logger.info("Starting pipeline — %d programs to process", len(run_list))
logger.info("=" * 60)

for idx, program in enumerate(run_list):
    logger.info("[%d/%d] %s", idx + 1, len(run_list), program["name"])
    logger.info("  URL: %s", program["url"])

    try:

        # --- Scrape ---
        try:
            full_text, dollar_lines = scrape_page(program["url"])
            logger.info("  Page fetched — %d chars | %d dollar mentions", len(full_text), len(dollar_lines))
        except Exception as e:
            raise RuntimeError(f"scrape_page failed: {e}")

        # --- OpenAI Extract ---
        try:
            extracted, tokens = extract_with_openai(program["name"], full_text, dollar_lines)
            logger.info("  OpenAI done — %d tokens | award_value: %s", tokens, extracted.get("award_value", "MISSING"))
        except Exception as e:
            raise RuntimeError(f"extract_with_openai failed: {e}")

        # --- Secondary URL Analysis ---
        secondary_links_analysis            = []
        total_tokens_for_secondary_analysis = 0

        secondary_urls = program["all_extracted_urls"][1:]
        if secondary_urls:
            logger.info("  Secondary URLs: %d", len(secondary_urls))
            for sec_url in secondary_urls:
                analysis = analyze_secondary_url(program["name"], program["url"], sec_url)
                secondary_links_analysis.append(analysis)
                total_tokens_for_secondary_analysis += analysis.get("tokens_used", 0)
                if analysis["status"] == "success":
                    logger.info("    -> %s | type: %s | %d tokens", sec_url, analysis["type"], analysis["tokens_used"])
                else:
                    logger.warning("    -> %s | FAILED: %s", sec_url, analysis["reason"])
                time.sleep(0.5)

        # --- Review Check ---
        # Flag only when the number of distinct award amounts is high enough that
        # the correct individual award value is genuinely ambiguous.
        # Threshold > 4 avoids false positives on programs with a few tiers/pools.
        award_amounts = extracted.get("award_amounts", [])
        award_value   = extracted.get("award_value", "Not specified")
        needs_review  = False
        review_reason = ""

        if len(award_amounts) > 4:
            needs_review  = True
            review_reason = f"{len(award_amounts)} dollar amounts found human should confirm correct award_value"
        elif award_value == "Varies" and len(award_amounts) > 1:
            needs_review  = True
            review_reason = "Award marked Varies but OpenAI found amounts verify which is individual award"
        elif len(award_amounts) > 1 and any(x in award_value for x in ["M", "B", "million", "billion"]):
            needs_review  = True
            review_reason = "Award value may be fund total not individual award verify manually"

        # --- Build Record ---
        final_record = {
            "id":                                 program["id"],
            "source_url":                         program["url"],
            "state":                              "Maryland",
            "title":                              extracted.get("title",                  program["name"]),
            "opportunity_type":                   extracted.get("opportunity_type",       program["type"]),
            "summary":                            extracted.get("summary",                "Not specified"),
            "description":                        extracted.get("description",            "Not specified"),
            "sponsor":                            extracted.get("sponsor",                "Not specified"),
            "sponsor_website":                    extracted.get("sponsor_website",        "Not specified"),
            "logo_url":                           extracted.get("logo_url",               "No logo URL found"),
            "direct_application_url":             extracted.get("direct_application_url", "Not specified"),
            "award_value":                        award_value,
            "cash_award":                         extracted.get("cash_award",             "Not specified"),
            "award_amounts":                      award_amounts,
            "date_posted":                        extracted.get("date_posted",            "Not specified"),
            "deadline":                           extracted.get("deadline",               "Not specified"),
            "rolling":                            extracted.get("rolling",                "No"),
            "global_opportunity":                 extracted.get("global_opportunity",     "No"),
            "location":                           extracted.get("location",               "Not specified"),
            "fee_required":                       extracted.get("fee_required",           "No"),
            "cost_to_participate":                extracted.get("cost_to_participate",    "No"),
            "equity_percentage":                  extracted.get("equity_percentage",      "No"),
            "safe_note":                          extracted.get("safe_note",              "No"),
            "contact_names":                      extracted.get("contact_names",          "Not specified"),
            "contact_email":                      extracted.get("contact_email",          "Not specified"),
            "contact_phone":                      extracted.get("contact_phone",          "Not specified"),
            "industry":                           extracted.get("industry",               "Not specified"),
            "tags":                               extracted.get("tags",          []),
            "areas_of_focus":                     extracted.get("areas_of_focus", []),
            "eligibility":                        extracted.get("eligibility",   []),
            "sdg_alignment":                      extracted.get("sdg_alignment", []),
            "needs_review":                       needs_review,
            "review_reason":                      review_reason,
            "dollar_scan":                        dollar_lines,
            "tokens_used":                        tokens,
            "related_links_analysis":             secondary_links_analysis,
            "tokens_used_for_secondary_analysis": total_tokens_for_secondary_analysis
        }

        if needs_review:
            flagged_records.append(final_record)
            logger.info("  FLAGGED: %s", review_reason)
        else:
            clean_records.append(final_record)
            logger.info("  Clean")

    except Exception as e:
        errors.append({"program": program["name"], "url": program["url"], "error": str(e)})
        logger.error("  ERROR — %s", e)

    time.sleep(1)

logger.info("=" * 60)
logger.info("Run complete")
logger.info("  Clean:   %d", len(clean_records))
logger.info("  Flagged: %d", len(flagged_records))
logger.info("  Errors:  %d", len(errors))
if errors:
    logger.warning("Errors encountered:")
    for e in errors:
        logger.warning("  - %s: %s", e["program"], e["error"])
logger.info("Log saved to: %s", log_filename)


##  Save Results to JSON

In [ ]:
output = {
    'clean':   clean_records,    # records ready for the database
    'flagged': flagged_records,  # records needing human review
    'errors':  errors            # programs that failed to process
}

# Write to JSON file
with open('maryland_full_results.json', 'w') as f:
    json.dump(output, f, indent=2)

# Count total tokens across all records (main + secondary)
total_tokens = sum(r.get('tokens_used', 0) for r in clean_records + flagged_records)
total_tokens += sum(r.get('tokens_used_for_secondary_analysis', 0) for r in clean_records + flagged_records)

logger.info("Saved to maryland_full_results.json")
logger.info("  Clean:        %d records", len(clean_records))
logger.info("  Flagged:      %d records", len(flagged_records))
logger.info("  Errors:       %d", len(errors))
logger.info("  Total tokens: %s", f"{total_tokens:,}")


## Review Flagged Records
Quick scan of everything that needs a human decision before final submission.

In [ ]:
if flagged_records:
    logger.info("%d records flagged for review:", len(flagged_records))
    for r in flagged_records:
        logger.info("  Program:       %s", r["title"])
        logger.info("  Reason:        %s", r["review_reason"])
        logger.info("  award_value:   %s", r["award_value"])
        logger.info("  award_amounts: %s", r["award_amounts"])
        logger.info("  URL:           %s", r["source_url"])
        logger.info("  ---")
else:
    logger.info("No flagged records — all clean!")


## Extract Secondary URL Records
Pulls all records where a secondary URL was analyzed and saves them to their own file.
Records typed as 'Related Program' or 'Other Funding Opportunity' are flagged with
needs_further_review  they may contain programs not on the orginal API.

In [ ]:
secondary_records = []  # one entry per secondary URL analyzed

all_records = clean_records + flagged_records  # search both for secondary URLs

for record in all_records:
    analyses = record.get('related_links_analysis', [])
    if not analyses:
        continue

    for analysis in analyses:
        secondary_records.append({
            'parent_program':           record['title'],
            'parent_url':               record['source_url'],
            'parent_opportunity_type':  record['opportunity_type'],
            'parent_award_value':       record['award_value'],
            'secondary_url':            analysis.get('url',     'Not specified'),
            'secondary_type':           analysis.get('type',    'Not specified'),
            'secondary_status':         analysis.get('status',  'Not specified'),
            'secondary_summary':        analysis.get('summary', 'Not specified'),
            'secondary_details':        analysis.get('details', {}),
            'tokens_used':              analysis.get('tokens_used', 0),
            'needs_further_review':     analysis.get('type') in ['Related Program', 'Other Funding Opportunity']
        })

with open('maryland_secondary_urls.json', 'w') as f:
    json.dump(secondary_records, f, indent=2)

logger.info("Secondary URL records extracted: %d", len(secondary_records))

for r in secondary_records:
    flag = "REVIEW — may contain additional opportunities" if r['needs_further_review'] else "Supplemental"
    logger.info("  Parent:   %s", r["parent_program"])
    logger.info("  Sec URL:  %s", r["secondary_url"])
    logger.info("  Type:     %s | %s", r["secondary_type"], flag)
    logger.info("  Summary:  %s", r["secondary_summary"])
    logger.info("  ---")